## Importing Libraries

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

load_dotenv()

llm = ChatOpenAI(model="gpt-5-nano", temperature=0.0)

### EVALUATION SCHEMA 

In [3]:
class EvalScore(BaseModel):
    score: int = Field(..., description="Score from 1-5. 5=perfect, 1=completely wrong")
    faithfulness: bool = Field(..., description="True if answer only uses provided context")
    reasoning: str = Field(..., description="One sentence explaining the score")


parser = JsonOutputParser(pydantic_object=EvalScore)

### LLM-AS-JUDGE EVALUATOR

In [6]:
eval_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert evaluator of AI assistant responses.
Evaluate the actual answer against the expected answer and context.

Scoring rubric:
5 = Correct, complete, only uses provided context
4 = Mostly correct with minor gaps
3 = Partially correct but missing key information
2 = Mostly wrong but has some relevant content
1 = Completely wrong or fabricated information

{format_instructions}"""),
    ("human", """Question: {question}

Context provided to AI: {context}

Expected answer: {expected}

Actual answer: {actual}

Evaluate the actual answer."""),
])

eval_chain = eval_prompt | llm | parser

### TEST CASES

In [8]:
context = """
PostgreSQL B-tree indexes are the default index type.
They support equality (=) and range queries (<, >, <=, >=).
Composite indexes cover multiple columns — leftmost column must appear in WHERE clause.
Use EXPLAIN ANALYZE to check if queries are using indexes.
"""

test_cases = [
    {
        "question": "What query operators does a B-tree index support?",
        "expected": "B-tree indexes support equality (=) and range operators (<, >, <=, >=).",
        "actual": "B-tree indexes support equality and range queries including =, <, >, <=, and >=.",
    },
    {
        "question": "What is a composite index?",
        "expected": "A composite index covers multiple columns and requires the leftmost column in the WHERE clause.",
        "actual": "A composite index is an index on multiple columns. The order of columns is critical.",
    },
    {
        "question": "What is the best database for machine learning?",
        "expected": "The context does not cover this topic.",
        "actual": "PostgreSQL is the best database for machine learning because of its vector extension.",
        # This should score low — hallucinated, not in context
    },
]

print("=== LLM-as-Judge Evaluation ===\n")

for i, test in enumerate(test_cases, 1):
    print(f"Test Case {i}: {test['question']}")
    result = eval_chain.invoke({
        "question": test["question"],
        "context": context,
        "expected": test["expected"],
        "actual": test["actual"],
        "format_instructions": parser.get_format_instructions(),
    })
    print(f" Score: {result['score']}/5")
    print(f" Faithfulness: {result['faithfulness']}")
    print(f" Reasoning: {result['reasoning']}\n")
    print("-" * 50 + "\n")

=== LLM-as-Judge Evaluation ===

Test Case 1: What query operators does a B-tree index support?
 Score: 5/5
 Faithfulness: True
 Reasoning: The answer states equality and range operators (=, <, >, <=, >=) for B-tree indexes, matching the expected answer and the provided context.

--------------------------------------------------

Test Case 2: What is a composite index?
 Score: 4/5
 Faithfulness: True
 Reasoning: The answer correctly notes that a composite index is on multiple columns and that the order of the columns matters, but it does not explicitly state that the leftmost column must appear in the WHERE clause, which is the key contextual detail.

--------------------------------------------------

Test Case 3: What is the best database for machine learning?
 Score: 2/5
 Faithfulness: False
 Reasoning: The answer asserts PostgreSQL is the best database for machine learning and cites a vector extension, which is not covered or supported by the provided context (which only mentions 

### SIMPLE HEURISTIC EVALUATION

In [9]:
print("=== Heuristic Evaluation (keyword overlap) ===\n")

def simple_eval(expected: str, actual: str) -> dict:
    """
    Simple keyword-based evaluation.
    Checks what fraction of expected keywords appear in the actual answer.
    Not as good as LLM-as-judge but zero API cost.
    """
    expected_keywords = set(expected.lower().split())
    actual_keywords = set(actual.lower().split())
    stop_words = {"the", "a", "an", "is", "are", "and", "or", "in", "of", "to", "for"}
    expected_keywords = expected_keywords - stop_words
    actual_keywords = actual_keywords - stop_words
    overlap = expected_keywords & actual_keywords
    score = (len(overlap) / len(expected_keywords)) if expected_keywords else 0
    return {
        "keyword_overlap_score": round(score, 2),
        "matched_keywords": list(overlap),
        "missing_keywords": list(expected_keywords - actual_keywords),
    }

for i, test in enumerate(test_cases[:2], 1):
    print(f"Test Case {i}: {test['question']}")
    result = simple_eval(test["expected"], test["actual"])
    print(f" Keyword Overlap Score: {result['keyword_overlap_score']}")
    print(f" Matched Keywords: {result['matched_keywords']}")
    print(f" Missing Keywords: {result['missing_keywords']}\n")
    print("-" * 50 + "\n")

=== Heuristic Evaluation (keyword overlap) ===

Test Case 1: What query operators does a B-tree index support?
 Keyword Overlap Score: 0.64
 Matched Keywords: ['indexes', '>,', 'range', '<=,', 'support', 'b-tree', 'equality']
 Missing Keywords: ['(<,', '>=).', 'operators', '(=)']

--------------------------------------------------

Test Case 2: What is a composite index?
 Keyword Overlap Score: 0.4
 Matched Keywords: ['multiple', 'columns', 'index', 'composite']
 Missing Keywords: ['covers', 'column', 'leftmost', 'where', 'clause.', 'requires']

--------------------------------------------------

